<a href="https://colab.research.google.com/github/NethmiAmasha/Deep-Learning-for-Vision/blob/main/EN4554_AS02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#EN4554 Assignment 2 on CNNs
----

Name: Pathirana P.N.A

Index No.: 220449F


### Task 1: Backpropagation

(a) Linear, ReLU, and softmax cross-entropy

In [ ]:
import torch

class Linear:
    """y = x @ W + b.  x: (N, in), W: (in, out), b: (out,)"""
    def __init__(self, in_features, out_features, dtype=torch.float32, generator=None):
        std = (2.0 / in_features) ** 0.5          # Kaiming (He) init, suits ReLU
        self.W = torch.randn(in_features, out_features, dtype=dtype, generator=generator) * std
        self.b = torch.zeros(out_features, dtype=dtype)
        self.dW = torch.zeros_like(self.W)
        self.db = torch.zeros_like(self.b)

    def forward(self, x):
        self.x = x                                # cache input for backward
        return x @ self.W + self.b

    def backward(self, dout):
        self.dW = self.x.T @ dout                 # (in, N) @ (N, out)
        self.db = dout.sum(dim=0)
        return dout @ self.W.T                    # gradient w.r.t. input


class ReLU:
    def forward(self, x):
        self.mask = (x > 0).to(x.dtype)           # cache where the input was positive
        return x * self.mask

    def backward(self, dout):
        return dout * self.mask


class SoftmaxCrossEntropy:
    """Takes raw logits (N, C) and integer labels (N,). Returns mean loss."""
    def forward(self, logits, y):
        shifted = logits - logits.max(dim=1, keepdim=True).values      # numerical stability
        log_probs = shifted - torch.logsumexp(shifted, dim=1, keepdim=True)
        self.probs = log_probs.exp()
        self.y = y
        N = logits.shape[0]
        return -log_probs[torch.arange(N), y].mean()

    def backward(self):
        N = self.probs.shape[0]
        dlogits = self.probs.clone()
        dlogits[torch.arange(N), self.y] -= 1.0
        return dlogits / N